## Crawl .xml from  Pubmed

In [24]:
import requests
import xml.etree.ElementTree as ET
import nltk
import re
import time
import ssl # Import the SSL module
from nltk.corpus import stopwords

In [25]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    # Python < 2.7.9 / 3.4.3 doesn't have this, so we do nothing.
    pass
else:
    # Create an unverified SSL context to bypass potential certificate issues.
    # This is often necessary on corporate networks or with certain firewalls.
    ssl._create_default_https_context = _create_unverified_https_context

# Now, we try to find the resource and download it if it's missing.
try:
    nltk.data.find('tokenizers/punkt')
except LookupError: # <-- CORRECTED EXCEPTION
    print("Downloading NLTK's 'punkt' tokenizer models...")
    nltk.download('punkt')
    print("Download complete.")

In [27]:
# --- Part 2: Configuration ---
# Define the search query and parameters for the PubMed API.
disease_list = [
    "cancer",
    "diabetes",
    "hypertension",
    '"alzheimer\'s disease"',
    "obesity",
    "asthma",
    '"chronic kidney disease"',
    '"heart failure"',
    "stroke",
    "arthritis",
    "influenza",
    "tuberculosis",
    "malaria",
    '"multiple sclerosis"',
    "sepsis"
]

# NEW: Automatically build the search query by joining the list with "OR"
SEARCH_QUERY = " OR ".join(disease_list)
MAX_ARTICLES = 5000  # Number of articles to fetch (be mindful of API limits)
YOUR_EMAIL = "chiunli2010@gmail.com" # NCBI asks for an email for identification

# Base URLs for NCBI E-utilities
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

print(f"Preparing to crawl PubMed for articles matching: '{SEARCH_QUERY}'")

# --- Part 3: Fetch Article IDs (PMIDs) ---
print(f"Step 1: Searching for up to {MAX_ARTICLES} article IDs...")

esearch_params = {
    "db": "pubmed",
    "term": SEARCH_QUERY,
    "retmax": MAX_ARTICLES,
    "usehistory": "y",
    "tool": "MyPubMedCrawler",
    "email": YOUR_EMAIL
}

Preparing to crawl PubMed for articles matching: 'cancer OR diabetes OR hypertension OR "alzheimer's disease" OR obesity OR asthma OR "chronic kidney disease" OR "heart failure" OR stroke OR arthritis OR influenza OR tuberculosis OR malaria OR "multiple sclerosis" OR sepsis'
Step 1: Searching for up to 5000 article IDs...


In [28]:
try:
    # Make the request to the esearch endpoint
    esearch_response = requests.get(ESEARCH_URL, params=esearch_params)
    esearch_response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    # Parse the XML response to get the list of PMIDs
    esearch_root = ET.fromstring(esearch_response.content)
    id_list = [id_elem.text for id_elem in esearch_root.findall(".//Id")]

    if not id_list:
        print("No articles found for the given query. Please try a different search term.")
        exit()

    print(f"Successfully found {len(id_list)} article IDs.")
    
    # Get the WebEnv and QueryKey needed for the next step
    web_env = esearch_root.find(".//WebEnv").text
    query_key = esearch_root.find(".//QueryKey").text

except requests.exceptions.RequestException as e:
    print(f"An error occurred during the search request: {e}")
    exit()
except ET.ParseError as e:
    print(f"Failed to parse the search response XML: {e}")
    exit()

Successfully found 5000 article IDs.


In [29]:
# --- Part 4 & 5 (REVISED): Fetch, Parse, and Process in Batches ---
print("\nStep 2 & 3: Fetching, parsing, and processing XML data in batches...")

# --- Configuration for Batching ---
BATCH_SIZE = 200 # Number of articles to fetch per request. A good starting point.
corpus = []      # The corpus will be built up incrementally.

# --- Loop Through All Article IDs in Batches ---
for start_index in range(0, len(id_list), BATCH_SIZE):
    # Determine the end index for the current batch
    end_index = min(start_index + BATCH_SIZE, len(id_list))
    
    # Get the sub-list of PMIDs for this batch
    batch_ids = id_list[start_index:end_index]
    pmids_string = ",".join(batch_ids)

    print(f"Processing batch: articles {start_index + 1} to {end_index}...")

    # --- Fetching ---
    efetch_params = {
        "db": "pubmed",
        "id": pmids_string, # Use the batch IDs directly
        # We don't need WebEnv/QueryKey when providing IDs directly
        "rettype": "xml",
        "retmode": "xml",
        "tool": "MyPubMedCrawler",
        "email": YOUR_EMAIL
    }

    try:
        # Be nice to the API: delay between each batch request
        time.sleep(0.4)
        efetch_response = requests.get(EFETCH_URL, params=efetch_params)
        efetch_response.raise_for_status()
        xml_data = efetch_response.text

    except requests.exceptions.RequestException as e:
        print(f"   - An error occurred fetching batch: {e}. Skipping this batch.")
        continue # Move to the next batch

    # --- Parsing and Tokenizing (for the current batch) ---
    try:
        root = ET.fromstring(xml_data)
        
        # Iterate over each article IN THIS BATCH'S XML
        for article in root.findall(".//PubmedArticle"):
            title_element = article.find(".//ArticleTitle")
            title = title_element.text if title_element is not None else ""

            abstract_texts = []
            for abstract_element in article.findall(".//Abstract/AbstractText"):
                if abstract_element.text:
                    abstract_texts.append(abstract_element.text)
            abstract = "\n".join(abstract_texts)
            
            full_text = f"{title}\n{abstract}"
            if not full_text.strip():
                continue

            sentences = nltk.sent_tokenize(full_text)
            for sentence in sentences:
                words = re.findall(r'\b[a-zA-Z]+\b', sentence.lower())
                if words:
                    corpus.append(words)

    except ET.ParseError as e:
        print(f"   - Failed to parse XML for this batch: {e}. Skipping.")
        continue

print(f"\nProcessing complete. Generated a corpus with {len(corpus)} sentences.")


# Your Part 6 and Part 7 (model training) can now proceed as before.
# ... (rest of your script)


Step 2 & 3: Fetching, parsing, and processing XML data in batches...
Processing batch: articles 1 to 200...
Processing batch: articles 201 to 400...
Processing batch: articles 401 to 600...
Processing batch: articles 601 to 800...
Processing batch: articles 801 to 1000...
Processing batch: articles 1001 to 1200...
Processing batch: articles 1201 to 1400...
Processing batch: articles 1401 to 1600...
Processing batch: articles 1601 to 1800...
Processing batch: articles 1801 to 2000...
Processing batch: articles 2001 to 2200...
Processing batch: articles 2201 to 2400...
Processing batch: articles 2401 to 2600...
Processing batch: articles 2601 to 2800...
Processing batch: articles 2801 to 3000...
Processing batch: articles 3001 to 3200...
Processing batch: articles 3201 to 3400...
Processing batch: articles 3401 to 3600...
Processing batch: articles 3601 to 3800...
Processing batch: articles 3801 to 4000...
Processing batch: articles 4001 to 4200...
Processing batch: articles 4201 to 440

In [30]:
print("\nSaving the corpus to a text file for future use...")

with open("pubmed_corpus.txt", "w", encoding="utf-8") as f:
    for sentence in corpus:
        f.write(" ".join(sentence) + "\n")

print("Corpus saved to 'pubmed_corpus.txt'")


Saving the corpus to a text file for future use...
Corpus saved to 'pubmed_corpus.txt'


## Construct Model

In [64]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
import multiprocessing
from nltk.stem import PorterStemmer
from collections import defaultdict
import json

### Preprocess

#### No_stop & Stemmed

In [48]:
print("--- Starting Corpus Pre-processing ---")
print("This script will create a new corpus file with stop words removed.")

# --- 1. Setup and NLTK Download ---
# Ensure the 'stopwords' resource is available.
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading NLTK 'stopwords' resource...")
    nltk.download('stopwords')

# --- 2. Define Files and Load Stop Words ---
input_corpus_path = "pubmed_corpus.txt"
output_corpus_path = "pubmed_corpus_no_stopwords.txt"

# Load stop words into a set for fast lookup
stop_words = set(stopwords.words('english'))
print(f"Loaded {len(stop_words)} English stop words.")

# --- 3. Read, Filter, and Write the New Corpus File ---
# This is a memory-efficient, line-by-line process.
line_count = 0
try:
    with open(input_corpus_path, 'r', encoding='utf-8') as infile, \
         open(output_corpus_path, 'w', encoding='utf-8') as outfile:
        
        print(f"Reading from '{input_corpus_path}' and writing to '{output_corpus_path}'...")
        
        for line in infile:
            words = line.strip().split()
            # Filter out the stop words
            filtered_words = [word for word in words if word.lower() not in stop_words]
            
            # Write the new, filtered sentence to the output file, if it's not empty
            if filtered_words:
                outfile.write(" ".join(filtered_words) + "\n")
            
            line_count += 1
            if line_count % 500000 == 0: # Print a progress update every 500k lines
                print(f"  ...processed {line_count:,} lines")

    print(f"\nSuccessfully processed {line_count:,} lines.")
    print(f"Filtered corpus saved to: '{output_corpus_path}'")

except FileNotFoundError:
    print(f"ERROR: The input file '{input_corpus_path}' was not found. Please make sure it's in the same directory.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Starting Corpus Pre-processing ---
This script will create a new corpus file with stop words removed.
Loaded 198 English stop words.
Reading from 'pubmed_corpus.txt' and writing to 'pubmed_corpus_no_stopwords.txt'...

Successfully processed 42,339 lines.
Filtered corpus saved to: 'pubmed_corpus_no_stopwords.txt'


In [49]:
input_corpus_path = "pubmed_corpus_no_stopwords.txt" # We use the stopword-free file as our source
output_corpus_path = "pubmed_corpus_stemmed.txt"

stemmer = PorterStemmer()
# We still need stop words to make sure we don't accidentally re-introduce them
stop_words = set(stopwords.words('english'))

print(f"Reading from '{input_corpus_path}'")
print(f"Writing stemmed output to '{output_corpus_path}'")

# --- 3. Read, Stem, and Write the New Corpus File ---
line_count = 0
try:
    with open(input_corpus_path, 'r', encoding='utf-8') as infile, \
         open(output_corpus_path, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            words = line.strip().split()
            
            # Apply stemming to each word in the line
            # (We don't need to check for stop words again, but it's good practice)
            stemmed_words = [stemmer.stem(word) for word in words if word not in stop_words]
            
            if stemmed_words:
                outfile.write(" ".join(stemmed_words) + "\n")
            
            line_count += 1
            if line_count % 500000 == 0:
                print(f"  ...processed {line_count:,} lines")

    print(f"\nSuccessfully processed and stemmed {line_count:,} lines.")
    print(f"Stemmed corpus saved to: '{output_corpus_path}'")

except FileNotFoundError:
    print(f"ERROR: The input file '{input_corpus_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Reading from 'pubmed_corpus_no_stopwords.txt'
Writing stemmed output to 'pubmed_corpus_stemmed.txt'

Successfully processed and stemmed 42,327 lines.
Stemmed corpus saved to: 'pubmed_corpus_stemmed.txt'


#### stem_map

In [65]:
print("--- Creating Stem-to-Originals Mapping ---")

# --- 1. Initialize Tools and Define Files ---
input_corpus_path = "pubmed_corpus_no_stopwords.txt"
output_map_path = "stem_map.json"

stemmer = PorterStemmer()

# defaultdict is perfect for this: if a key doesn't exist, it creates it with a default value (a set).
# We use a set to automatically handle duplicate words.
stem_map = defaultdict(set)

# --- 2. Read Corpus and Build the Map ---
line_count = 0
word_count = 0
print(f"Reading corpus from: {input_corpus_path}")
try:
    with open(input_corpus_path, 'r', encoding='utf-8') as infile:
        for line in infile:
            words = line.strip().split()
            for word in words:
                stem = stemmer.stem(word)
                stem_map[stem].add(word)
                word_count += 1
            
            line_count += 1
            if line_count % 500000 == 0:
                print(f"  ...processed {line_count:,} lines ({word_count:,} words)")

    print(f"\nFinished processing. Found {len(stem_map):,} unique stems.")

    # --- 3. Convert Sets to Lists and Save to JSON ---
    # JSON cannot serialize sets, so we convert them to lists.
    final_map = {stem: sorted(list(originals)) for stem, originals in stem_map.items()}

    print(f"Saving the final map to: {output_map_path}")
    with open(output_map_path, 'w', encoding='utf-8') as outfile:
        json.dump(final_map, outfile, indent=2)
    
    print("--- Mapping complete! ---")

except FileNotFoundError:
    print(f"ERROR: The input file '{input_corpus_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Creating Stem-to-Originals Mapping ---
Reading corpus from: pubmed_corpus_no_stopwords.txt

Finished processing. Found 19,366 unique stems.
Saving the final map to: stem_map.json
--- Mapping complete! ---


#### Model Set

In [34]:
print("\n--- Starting Word2Vec Model Training ---")

# --- Model Configuration ---
# These are common parameters you can tune:
# vector_size: The dimensionality of the word vectors. Larger can capture more info, but requires more data.
# window: The maximum distance between the current and predicted word within a sentence.
# min_count: Ignores all words with a total frequency lower than this.
# workers: Number of CPU cores to use for training. Let's use all available cores.
VECTOR_SIZE = 100
WINDOW_SIZE = 5
MIN_WORD_COUNT = 5 # A word must appear at least 5 times to be included
WORKERS = multiprocessing.cpu_count()


--- Starting Word2Vec Model Training ---


In [56]:
corpus_file = "pubmed_corpus_stemmed.txt"
print(f"Streaming sentences from the file: {corpus_file}")

Streaming sentences from the file: pubmed_corpus_stemmed.txt


### CBOW Model

In [57]:
print("\nStep 7.1: Training the CBOW model...")

sentences = LineSentence(corpus_file)
# For CBOW, the key parameter is sg=0
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=MIN_WORD_COUNT,
    workers=WORKERS,
    sg=0  # sg=0 specifies the CBOW architecture
)

# Save the trained model to a file. This file contains all the "weights" (vectors).
cbow_model.save("pubmed_cbow_stemmed.model")
print("CBOW model trained and saved to 'pubmed_cbow_stemmed.model'")


Step 7.1: Training the CBOW model...
CBOW model trained and saved to 'pubmed_cbow_stemmed.model'


### Skip-Gram Model

In [58]:
print("\nStep 4.1: Training the Skip-gram model...")
# IMPORTANT: Because 'sentences' is an iterator, it can only be used once.
# We must re-initialize it to read the file from the beginning again.
sentences_for_sg = LineSentence(corpus_file)

skipgram_model = Word2Vec(
    sentences=sentences_for_sg,
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=MIN_WORD_COUNT,
    workers=WORKERS,
    sg=1  # sg=1 for Skip-gram
)

# Save the trained model.
skipgram_model.save("pubmed_skipgram_stemmed.model")
print("Skip-gram model trained and saved to 'pubmed_skipgram_stemmed.model'")

print("\n--- Model Training Complete ---")


Step 4.1: Training the Skip-gram model...
Skip-gram model trained and saved to 'pubmed_skipgram_stemmed.model'

--- Model Training Complete ---


In [60]:
# --- 5. Test the newly trained models (Example Usage) ---
print("\n--- Testing Model Similarity ---")

try:
    test_word = "patient"
    
    # Load and test the CBOW model
    loaded_cbow = Word2Vec.load("pubmed_cbow_no_stopwords.model")
    similar_cbow = loaded_cbow.wv.most_similar(test_word, topn=10)
    print(f"\nWords most similar to '{test_word}' using CBOW model:")
    for word, score in similar_cbow:
        print(f"  - {word}: {score:.4f}")

    # Load and test the Skip-gram model
    loaded_sg = Word2Vec.load("pubmed_skipgram_no_stopwords.model")
    similar_sg = loaded_sg.wv.most_similar(test_word, topn=10)
    print(f"\nWords most similar to '{test_word}' using Skip-gram model:")
    for word, score in similar_sg:
        print(f"  - {word}: {score:.4f}")

    loaded_sg = Word2Vec.load("pubmed_cbow_stemmed.model")
    similar_sg = loaded_sg.wv.most_similar(test_word, topn=10)
    print(f"\nWords most similar to '{test_word}' using Skip-gram model:")
    for word, score in similar_sg:
        print(f"  - {word}: {score:.4f}")

    loaded_sg = Word2Vec.load("pubmed_skipgram_stemmed.model")
    similar_sg = loaded_sg.wv.most_similar(test_word, topn=10)
    print(f"\nWords most similar to '{test_word}' using Skip-gram model:")
    for word, score in similar_sg:
        print(f"  - {word}: {score:.4f}")

except KeyError:
    print(f"The word '{test_word}' is not in the models' vocabulary.")
except Exception as e:
    print(f"An error occurred during testing: {e}")


--- Testing Model Similarity ---

Words most similar to 'patient' using CBOW model:
  - surgical: 0.7789
  - oncological: 0.7433
  - local: 0.7381
  - improve: 0.7367
  - centered: 0.7354
  - pros: 0.7307
  - improving: 0.7272
  - better: 0.7272
  - procedural: 0.7259
  - optimize: 0.7250

Words most similar to 'patient' using Skip-gram model:
  - careful: 0.6061
  - individual: 0.5689
  - pros: 0.5604
  - details: 0.5506
  - participant: 0.5444
  - centered: 0.5443
  - instrument: 0.5427
  - consideration: 0.5382
  - sic: 0.5374
  - clinician: 0.5306

Words most similar to 'patient' using Skip-gram model:
  - surgeri: 0.6557
  - pmp: 0.6447
  - locoregion: 0.6404
  - ofa: 0.6321
  - primari: 0.6319
  - secondari: 0.6315
  - symptomat: 0.6296
  - flot: 0.6295
  - moh: 0.6180
  - recurr: 0.6107

Words most similar to 'patient' using Skip-gram model:
  - flot: 0.6467
  - individu: 0.6418
  - octogenarian: 0.6368
  - poaf: 0.6279
  - nste: 0.6233
  - hepatectomi: 0.6202
  - pwh: 0.6177
 